# TCC 05 — Análise pós-hoc de 2024

Este notebook não retreina nem substitui o modelo congelado. Ele verifica os hashes da execução final, audita erros, compara limiares de forma exploratória e extrai os coeficientes do Poisson.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Baixar a versão aprovada do repositório

In [ ]:
from pathlib import Path
import shutil
import subprocess

REPO_DIR = Path('/content/tcc-vigilancia-wbe-iot')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run([
    'git', 'clone', '--branch', 'feature/pipeline-modelagem', '--single-branch',
    'https://github.com/samuelfzst-bit/tcc-vigilancia-wbe-iot.git', str(REPO_DIR)
], check=True)
print(subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip())

## Instalar versões compatíveis com os modelos salvos

A avaliação final utilizou scikit-learn 1.6.1; a mesma versão é instalada para leitura segura dos arquivos `.joblib`.

In [ ]:
%pip install -q 'scikit-learn==1.6.1' 'pandas==2.2.3' 'numpy==2.1.3' 'matplotlib>=3.8,<4' 'joblib>=1.4,<2'

## Validar caminhos e executar a análise

O diretório de saída é separado da execução confirmatória. Nada é escrito dentro da pasta congelada.

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/TCC')
RUN_ID = '20261016T035301Z'
RUN_DIR = DRIVE_ROOT / 'final_evaluation' / RUN_ID
OUTPUT_DIR = DRIVE_ROOT / 'posthoc_2024' / RUN_ID

required = [
    'manifest_final.json', '00_protocolo_congelado.json',
    '01_metricas_teste_2024.csv', '03_previsoes_teste_2024.csv',
    '05_modelo_poisson_t1.joblib', '05_modelo_poisson_t2.joblib',
]
missing = [name for name in required if not (RUN_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Arquivos ausentes na execução final: {missing}')

subprocess.run([
    'python', str(REPO_DIR / 'scripts/analyze_posthoc_2024.py'),
    '--run-dir', str(RUN_DIR),
    '--output-dir', str(OUTPUT_DIR),
], check=True, cwd=REPO_DIR)
print(f'Resultados salvos em: {OUTPUT_DIR}')

## Conferir o relatório e os gráficos

In [ ]:
from IPython.display import Markdown, display
display(Markdown((OUTPUT_DIR / 'RELATORIO_POS_HOC_2024.md').read_text(encoding='utf-8')))

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

for filename in [
    '06_erros_semanais_2024.png',
    '07_coeficientes_poisson.png',
    '08_sensibilidade_limiares.png',
]:
    image = Image.open(OUTPUT_DIR / filename)
    plt.figure(figsize=(15, 8))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

## Artefato preparado para o dashboard

O arquivo `05_previsoes_dashboard_2024.csv` é somente um contrato de carga. A criação ou alteração da tabela no Supabase deve ocorrer depois da conferência de chaves e relacionamentos existentes.

In [ ]:
import pandas as pd
dashboard = pd.read_csv(OUTPUT_DIR / '05_previsoes_dashboard_2024.csv')
display(dashboard.head())
print('Dimensão:', dashboard.shape)
print('Duplicidades da chave proposta:', dashboard.duplicated(['versao_modelo', 'horizonte', 'inicio_semana_alvo']).sum())